# Week 12 Capstone — validated GP-UCB proposals

## tl;dr

This notebook reconstructs the post-Week-11 dataset from the immutable 88-row canonical ledger, validates the required observation counts, and generates one deterministic, bounded, non-duplicate BBO proposal per function. No Week 12 returned outputs are available, so the proposals are not observations.

## Table of contents

1. [Overview](#overview)
2. [Objectives](#objectives)
3. [Evidence provenance](#evidence-provenance)
4. [Environment and setup](#environment-setup)
5. [Data validation](#data-validation)
6. [Descriptive EDA](#descriptive-eda)
7. [Visual EDA](#visual-eda)
8. [GP model configuration](#gp-model-configuration)
9. [GP-UCB values](#gp-ucb-values)
10. [Reproducibility checks](#reproducibility-checks)
11. [Conclusions and next steps](#conclusions-next-steps)


<a id="overview"></a>
## 1. Overview

Canonical all-function weekly analysis using the evidence and executed results already present on main.

<a id="objectives"></a>
## 2. Objectives

Validate F1–F8, review the existing EDA, and report the preserved GP-UCB values without changing calculations or evidence.

<a id="evidence-provenance"></a>
## 3. Evidence provenance

## Context & Methods

The source of truth is `Results/query_output_ledger.csv` plus the pristine starter arrays under `Week_01/Function_XX/03_Data`. Each function uses a Matérn-5/2 Gaussian Process with target normalisation and UCB (`kappa=0.1`). Twenty thousand candidates are generated from a fixed per-function seed.

### Key assumptions

- The canonical ledger is correctly paired and immutable.
- Historical observations are validated over `[0, 1]^d`.
- New portal submissions are restricted to `[0.000000, 0.999999]^d`.
- Six-decimal duplicate and format checks match submission precision.

<a id="environment-setup"></a>
## 4. Environment and setup

Import the existing utilities and locate the repository data sources.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'Results' / 'query_output_ledger.csv').is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Repository root not found')
sys.path.insert(0, str(ROOT))

from Code.data_loading import load_starter_data, append_observations
from Code.gaussian_process import fit_gaussian_process, predict_with_uncertainty
from Code.candidate_generation import make_rng, uniform_candidates
from Code.query_selection import select_query
from Code.portal_format import (
    SUBMISSION_LOWER_BOUND, SUBMISSION_UPPER_BOUND,
    format_portal_query, validate_portal_query,
)
from Code.weekly_evidence import DIMENSIONS, pairs_through_week

LEDGER = ROOT / 'Results' / 'query_output_ledger.csv'
CHECKSUM = ROOT / 'Results' / 'query_output_ledger.sha256'
expected_hash = CHECKSUM.read_text().split()[0]
actual_hash = hashlib.sha256(LEDGER.read_bytes()).hexdigest()
assert actual_hash == expected_hash
print(f'Canonical ledger checksum verified: {actual_hash}')

Canonical ledger checksum verified: 3cabc618a9b9b4a7b069309133a32850384f2f6eaad51c4264a9efb6cc5e874f


<a id="data-validation"></a>
## 5. Data validation

## Data

### 1. Reconstruct and validate cumulative observations

In [2]:
EXPECTED_COUNTS = {1: 21, 2: 21, 3: 26, 4: 41, 5: 31, 6: 31, 7: 41, 8: 51}
datasets = {}
validation_rows = []
for function in range(1, 9):
    X, y = load_starter_data(function, repository_root=ROOT)
    pairs = pairs_through_week(11, function)
    new_X = np.asarray([query for query, _ in pairs], dtype=float)
    new_y = np.asarray([value for _, value in pairs], dtype=float)
    X, y = append_observations(X, y, new_X, new_y)
    assert X.shape == (EXPECTED_COUNTS[function], DIMENSIONS[function])
    assert y.shape == (EXPECTED_COUNTS[function],)
    assert np.isfinite(X).all() and np.isfinite(y).all()
    assert ((X >= 0) & (X <= 1)).all()
    datasets[function] = (X, y)
    validation_rows.append({'Function': f'F{function}', 'Dimensions': X.shape[1], 'Observations': len(X), 'Best observed': float(np.max(y))})
validation = pd.DataFrame(validation_rows)
validation

,Function,Dimensions,Observations,Best observed
0,F1,2,21,7.710875e-16
1,F2,2,21,6.112052e-01
2,F3,3,26,-3.483531e-02
3,F4,4,41,-1.981075e+00
4,F5,4,31,1.465512e+03
5,F6,5,31,-7.142649e-01
6,F7,6,41,2.149905e+00
7,F8,8,51,9.939904e+00


<a id="descriptive-eda"></a>
## 6. Descriptive EDA

Review the descriptive summaries generated from the validated function datasets.

<a id="visual-eda"></a>
## 7. Visual EDA

### 2. Export validated figures by function

For every function, this section displays data-quality checks, summary statistics, and six figures. Each figure is also written as an individual PNG under `Week_12/Function_XX/05_Figures`. Orange points are canonical ledger returns; starter observations are blue.

In [3]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

figure_manifest = []
for function, (X, y) in datasets.items():
    starter_count = len(y) - 11
    query_index = np.arange(1, len(y) + 1)
    best_so_far = np.maximum.accumulate(y)
    figure_dir = ROOT / 'Week_12' / f'Function_{function:02d}' / '05_Figures'
    figure_dir.mkdir(parents=True, exist_ok=True)

    display(Markdown(f'#### Function {function:02d}'))
    quality = pd.DataFrame([{
        'input_shape': str(X.shape), 'output_shape': str(y.shape),
        'missing_values': int(np.isnan(X).sum() + np.isnan(y).sum()),
        'infinite_values': int(np.isinf(X).sum() + np.isinf(y).sum()),
        'out_of_bounds_inputs': int(((X < 0) | (X > 1)).sum()),
        'duplicate_input_rows_6dp': int(len(X) - len(np.unique(np.round(X, 6), axis=0))),
        'best_observation': int(np.argmax(y) + 1), 'best_output': float(np.max(y)),
    }])
    display(Markdown('**Data-quality checks**')); display(quality)
    summary_frame = pd.DataFrame(X, columns=[f'x{i + 1}' for i in range(X.shape[1])])
    summary_frame['objective'] = y
    display(Markdown('**Summary statistics**')); display(summary_frame.describe().T)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(query_index, y, marker='o', linewidth=1.3, label='objective')
    ax.scatter(query_index[starter_count:], y[starter_count:], color='darkorange', label='canonical return', zorder=3)
    ax.set(title=f'Week 12 Function {function:02d} — objective trace', xlabel='Verified observation', ylabel='Objective')
    ax.legend(); fig.tight_layout()
    path = figure_dir / f'week_12_function_{function:02d}_objective_trace.png'
    fig.savefig(path, dpi=160, bbox_inches='tight'); display(fig); plt.close(fig)
    figure_manifest.append({'function': function, 'figure': 'objective_trace', 'path': str(path.relative_to(ROOT))})

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.step(query_index, best_so_far, where='post', color='crimson', linewidth=1.8)
    ax.set(title=f'Week 12 Function {function:02d} — verified best so far', xlabel='Verified observation', ylabel='Best objective')
    fig.tight_layout()
    path = figure_dir / f'week_12_function_{function:02d}_best_so_far.png'
    fig.savefig(path, dpi=160, bbox_inches='tight'); display(fig); plt.close(fig)
    figure_manifest.append({'function': function, 'figure': 'best_so_far', 'path': str(path.relative_to(ROOT))})

    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.55 * X.shape[1])))
    image = ax.imshow(X.T, aspect='auto', cmap='viridis', vmin=0, vmax=1)
    ax.set(title=f'Week 12 Function {function:02d} — input coordinates', xlabel='Verified observation', ylabel='Dimension')
    ax.set_yticks(range(X.shape[1]), [f'x{i + 1}' for i in range(X.shape[1])])
    fig.colorbar(image, ax=ax, label='Coordinate value'); fig.tight_layout()
    path = figure_dir / f'week_12_function_{function:02d}_input_heatmap.png'
    fig.savefig(path, dpi=160, bbox_inches='tight'); display(fig); plt.close(fig)
    figure_manifest.append({'function': function, 'figure': 'input_heatmap', 'path': str(path.relative_to(ROOT))})

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(y, bins=min(12, max(6, int(np.sqrt(len(y))))), color='#4472C4', edgecolor='#243447', alpha=0.85)
    ax.axvline(np.mean(y), color='#D97706', linestyle='--', label=f'mean = {np.mean(y):.4g}')
    ax.axvline(np.median(y), color='#7A284E', linestyle=':', label=f'median = {np.median(y):.4g}')
    ax.set(title=f'Week 12 Function {function:02d} — objective distribution', xlabel='Objective', ylabel='Frequency'); ax.legend(); fig.tight_layout()
    path = figure_dir / f'week_12_function_{function:02d}_output_distribution.png'
    fig.savefig(path, dpi=160, bbox_inches='tight'); display(fig); plt.close(fig)
    figure_manifest.append({'function': function, 'figure': 'output_distribution', 'path': str(path.relative_to(ROOT))})

    columns = min(3, X.shape[1]); rows = int(np.ceil(X.shape[1] / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3 * rows), squeeze=False)
    for dimension, ax in enumerate(axes.flat):
        if dimension < X.shape[1]:
            ax.hist(X[:, dimension], bins=10, range=(0, 1), color='#4472C4', edgecolor='#243447', alpha=0.85)
            ax.set(title=f'x{dimension + 1}', xlabel='Coordinate', ylabel='Frequency', xlim=(0, 1))
        else:
            ax.axis('off')
    fig.suptitle(f'Week 12 Function {function:02d} — input distributions')
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    path = figure_dir / f'week_12_function_{function:02d}_input_distributions.png'
    fig.savefig(path, dpi=160, bbox_inches='tight'); display(fig); plt.close(fig)
    figure_manifest.append({'function': function, 'figure': 'input_distributions', 'path': str(path.relative_to(ROOT))})

    correlation = summary_frame.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(max(5, 0.8 * len(correlation)), max(4, 0.7 * len(correlation))))
    image = ax.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
    labels = correlation.columns.tolist()
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right')
    ax.set_yticks(range(len(labels)), labels)
    ax.set_title(f'Week 12 Function {function:02d} — Pearson correlation')
    fig.colorbar(image, ax=ax, label='Correlation', shrink=0.85); fig.tight_layout()
    path = figure_dir / f'week_12_function_{function:02d}_correlation_heatmap.png'
    fig.savefig(path, dpi=160, bbox_inches='tight'); display(fig); plt.close(fig)
    figure_manifest.append({'function': function, 'figure': 'correlation_heatmap', 'path': str(path.relative_to(ROOT))})

figure_manifest = pd.DataFrame(figure_manifest)
figure_manifest

#### Function 01

**Data-quality checks**

,input_shape,output_shape,missing_values,infinite_values,out_of_bounds_inputs,duplicate_input_rows_6dp,best_observation,best_output
0,"(21, 2)","(21,)",0,0,0,0,3,7.710875e-16


**Summary statistics**

,count,mean,std,min,25%,50%,75%,max
x1,21.0,0.519454,0.234471,0.082507,3.532930e-01,4.214100e-01,6.834182e-01,8.977140e-01
x2,21.0,0.609578,0.343480,0.071186,3.215980e-01,7.329999e-01,9.195310e-01,9.996730e-01
objective,21.0,-0.000172,0.000787,-0.003606,-3.089424e-96,7.651210e-239,1.322677e-79,7.710875e-16


...

<Figure size 900x440 with 2 Axes>

<Figure size 700x450 with 1 Axes>

<Figure size 1200x900 with 9 Axes>

<Figure size 720x630 with 2 Axes>

,function,figure,path
0,1,objective_trace,Week_12/Function_01/05_Figures/week_12_functio...
1,1,best_so_far,Week_12/Function_01/05_Figures/week_12_functio...
2,1,input_heatmap,Week_12/Function_01/05_Figures/week_12_functio...
3,1,output_distribution,Week_12/Function_01/05_Figures/week_12_functio...
4,1,input_distributions,Week_12/Function_01/05_Figures/week_12_functio...
5,1,correlation_heatmap,Week_12/Function_01/05_Figures/week_12_functio...
6,2,objective_trace,Week_12/Function_02/05_Figures/week_12_functio...
7,2,best_so_far,Week_12/Function_02/05_Figures/week_12_functio...
8,2,input_heatmap,Week_12/Function_02/05_Figures/week_12_functio...
9,2,output_distribution,Week_12/Function_02/05_Figures/week_12_functio...


<a id="gp-model-configuration"></a>
## 8. GP model configuration

Retain the existing Gaussian Process configuration and deterministic candidate-generation settings from main.

<a id="gp-ucb-values"></a>
## 9. GP-UCB values

## Results

### 3. Fit GP-UCB models and select new queries

In [4]:
proposal_rows = []
for function, (X, y) in datasets.items():
    model = fit_gaussian_process(X, y, optimizer_restarts=3, random_state=4200 + function)
    candidates = uniform_candidates(DIMENSIONS[function], 20_000, rng=make_rng(4200 + function))
    mean, std = predict_with_uncertainty(model, candidates)
    selected = select_query(candidates, X, mean, std, method='ucb', kappa=0.1, decimals=6)
    duplicate = np.any(np.all(np.round(X, 6) == selected.query, axis=1))
    assert not duplicate and ((selected.query >= SUBMISSION_LOWER_BOUND) & (selected.query <= SUBMISSION_UPPER_BOUND)).all()
    submission_query = format_portal_query(selected.query, dimensions=DIMENSIONS[function])
    validate_portal_query(submission_query, dimensions=DIMENSIONS[function])
    proposal_rows.append({
        'week': 12, 'function': function, 'dimensions': DIMENSIONS[function],
        'observation_count': len(X), 'query': json.dumps(selected.query.tolist(), separators=(',', ':')),
        'submission_query': submission_query,
        'predicted_mean': selected.predicted_mean, 'predictive_std': selected.predicted_std,
        'ucb_score': selected.acquisition, 'kernel': str(model.kernel_),
        'candidate_count': 20_000, 'kappa': 0.1, 'random_seed': 4200 + function,
        'duplicate_at_6dp': duplicate, 'status': 'proposal_only_return_unavailable'
    })
proposals = pd.DataFrame(proposal_rows)
proposals[['function', 'observation_count', 'submission_query', 'predictive_std', 'ucb_score']]

/Users/john-peteramewu/Documents/Codex/2026-08-07/understand-the-project-explain-the-structure/work/publish-week11.wwIJ2I/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/john-peteramewu/Documents/Codex/2026-08-07/understand-the-project-explain-the-structure/work/publish-week11.wwIJ2I/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/john-peteramewu/Documents/Codex/2026-08-07/understand-the-project-explain-the-structure/work/publish-week11.wwIJ2I/.venv/lib/python3.12/site-packages/sklear

/Users/john-peteramewu/Documents/Codex/2026-08-07/understand-the-project-explain-the-structure/work/publish-week11.wwIJ2I/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/john-peteramewu/Documents/Codex/2026-08-07/understand-the-project-explain-the-structure/work/publish-week11.wwIJ2I/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/john-peteramewu/Documents/Codex/2026-08-07/understand-the-project-explain-the-structure/work/publish-week11.wwIJ2I/.venv/lib/python3.12/site-packages/sklear

,function,observation_count,submission_query,predictive_std,ucb_score
0,1,21,0.997914-0.748997,0.000284,0.000344
1,2,21,0.692504-0.619197,0.048199,0.628033
2,3,26,0.737864-0.713915-0.373523,0.041327,-0.019101
3,4,41,0.370439-0.382408-0.425132-0.390887,0.544867,-1.298079
4,5,31,0.099220-0.956522-0.978677-0.993261,165.074217,1847.642258
5,6,31,0.542093-0.205654-0.578843-0.946831-0.036709,0.139662,-0.464115
6,7,41,0.108716-0.301228-0.470084-0.228030-0.331783-0...,0.113192,2.111765
7,8,51,0.123374-0.097970-0.065526-0.059094-0.757316-0...,0.123934,9.962202


<a id="reproducibility-checks"></a>
## 10. Reproducibility checks

### 4. Save the proposal ledger and submission file

In [5]:
proposal_path = ROOT / 'Results' / 'bbo_query_ledger.csv'
archive_path = ROOT / 'Results' / 'archive' / 'bbo_query_ledger_kappa_2.0.csv'
if proposal_path.is_file():
    previous = pd.read_csv(proposal_path)
    if len(previous) == 8 and previous['kappa'].eq(2.0).all():
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        previous.to_csv(archive_path, index=False)
proposals.to_csv(proposal_path, index=False)
if archive_path.is_file():
    baseline = pd.read_csv(archive_path)
    comparison = baseline[['function', 'submission_query', 'predicted_mean', 'predictive_std', 'ucb_score']].merge(
        proposals[['function', 'submission_query', 'predicted_mean', 'predictive_std', 'ucb_score']],
        on='function', suffixes=('_kappa_2.0', '_kappa_0.1'))
    comparison.to_csv(ROOT / 'Results' / 'bbo_query_kappa_comparison.csv', index=False)
query_path = ROOT / 'Week_12' / '01_Queries' / 'week_12_query_points.txt'
query_path.write_text(''.join(f"Function_{row.function}:{row.submission_query}\n" for row in proposals.itertuples()), encoding='utf-8')
print(query_path.read_text())
comparison

Function_1:0.997914-0.748997
Function_2:0.692504-0.619197
Function_3:0.737864-0.713915-0.373523
Function_4:0.370439-0.382408-0.425132-0.390887
Function_5:0.099220-0.956522-0.978677-0.993261
Function_6:0.542093-0.205654-0.578843-0.946831-0.036709
Function_7:0.108716-0.301228-0.470084-0.228030-0.331783-0.836497
Function_8:0.123374-0.097970-0.065526-0.059094-0.757316-0.290599-0.242285-0.654268



,function,submission_query_kappa_2.0,predicted_mean_kappa_2.0,predictive_std_kappa_2.0,ucb_score_kappa_2.0,submission_query_kappa_0.1,predicted_mean_kappa_0.1,predictive_std_kappa_0.1,ucb_score_kappa_0.1
0,1,0.985530-0.508068,0.000019,0.000892,0.001804,0.997914-0.748997,0.000315,0.000284,0.000344
1,2,0.698570-0.005967,0.582219,0.088871,0.759961,0.692504-0.619197,0.623213,0.048199,0.628033
2,3,0.972188-0.997743-0.391105,-0.033628,0.071284,0.108940,0.737864-0.713915-0.373523,-0.023233,0.041327,-0.019101
3,4,0.370439-0.382409-0.425133-0.390888,-1.352565,0.544867,-0.262830,0.370439-0.382408-0.425132-0.390887,-1.352566,0.544867,-1.298079
4,5,0.874157-0.993155-0.978150-0.979670,1647.859676,307.051221,2261.962117,0.099220-0.956522-0.978677-0.993261,1831.134837,165.074217,1847.642258
5,6,0.004840-0.033276-0.614036-0.992733-0.018278,-0.812014,0.436965,0.061916,0.542093-0.205654-0.578843-0.946831-0.036709,-0.478081,0.139662,-0.464115
6,7,0.087492-0.296104-0.894458-0.260860-0.263170-0...,1.950674,0.214878,2.380431,0.108716-0.301228-0.470084-0.228030-0.331783-0...,2.100446,0.113192,2.111765
7,8,0.047423-0.249478-0.266938-0.124120-0.525139-0...,9.773742,0.259604,10.292949,0.123374-0.097970-0.065526-0.059094-0.757316-0...,9.949808,0.123934,9.962202


<a id="conclusions-next-steps"></a>
## 11. Conclusions and next steps

## Takeaways

- The canonical post-Week-11 counts are `21, 21, 26, 41, 31, 31, 41, 51`.
- Every proposed point is finite, within bounds, correctly dimensioned, and distinct from observed points at six-decimal precision.
- These are Week 12 submission proposals only. They must not be appended to the query/output ledger until authoritative returned outputs are available.